In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import csv
import os
from datetime import datetime


In [5]:
CSV_FILE = pd.read_csv("electricity_usage.csv")
THRESHOLD = 400   # monthly units threshold

In [6]:
print(CSV_FILE)

     Room        Date  Units
0     101  2025-07-01     20
1     102  2025-07-01     13
2     103  2025-07-01     11
3     104  2025-07-01     11
4     105  2025-07-01     11
..    ...         ...    ...
150   101  2025-07-31     15
151   102  2025-07-31     11
152   103  2025-07-31     13
153   104  2025-07-31      7
154   105  2025-07-31     13

[155 rows x 3 columns]


In [24]:
def log_usage():
    """Add a new record into the CSV file"""
    room = input("Enter Room No: ")
    date = input("Enter Date (YYYY-MM-DD): ")
    units = input("Enter Units Used: ")

    try:
        datetime.strptime(date, "%Y-%m-%d")  # validate date
        units = int(units)
    except ValueError:
        print("❌ Invalid input! Please try again.")
        return

    with open(CSV_FILE, "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([room, date, units])
    print("✅ Entry saved!")


In [25]:
def analyze_usage():
    """Show monthly usage totals and plot charts"""
    df = pd.read_csv(CSV_FILE)

    # convert types
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Units"] = pd.to_numeric(df["Units"], errors="coerce")

    # add Month column
    df["Month"] = df["Date"].dt.to_period("M")

    # group by Room + Month
    monthly = df.groupby(["Month", "Room"])["Units"].sum().unstack()

    print("\n📊 Monthly Usage Summary:")
    print(monthly)

    # plot each month
    for month in monthly.index:
        usage = monthly.loc[month]
        colors = ["red" if val > THRESHOLD else "blue" for val in usage]

        usage.plot(kind="bar", color=colors)
        plt.axhline(THRESHOLD, color="green", linestyle="--", label=f"Threshold {THRESHOLD}")
        plt.xlabel("Room")
        plt.ylabel("Total Units")
        plt.title(f"Hostel Electricity Usage - {month}")
        plt.legend()
        plt.show()


In [26]:
def room_report():
    """Detailed report for a specific room"""
    df = pd.read_csv(CSV_FILE)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    room = input("Enter Room No: ")
    room_data = df[df["Room"] == int(room)]

    if room_data.empty:
        print("❌ No data for this room.")
        return

    total_units = room_data["Units"].sum()
    avg_units = room_data["Units"].mean()
    peak_units = room_data["Units"].max()
    peak_date = room_data.loc[room_data["Units"].idxmax(), "Date"]

    print(f"\n📑 Report for Room {room}")
    print(f"Total Units: {total_units}")
    print(f"Average Daily Units: {avg_units:.2f}")
    print(f"Peak Usage: {peak_units} units on {peak_date.date()}")


In [27]:
def export_report():
    """Export monthly summary to CSV"""
    df = pd.read_csv(CSV_FILE)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Month"] = df["Date"].dt.to_period("M")

    monthly = df.groupby(["Month", "Room"])["Units"].sum().reset_index()
    monthly.to_csv("monthly_report.csv", index=False)
    print("📂 Report saved as monthly_report.csv")


In [28]:
def main():
    while True:
        print("\n--- Hostel Electricity Usage Logger ---")
        print("1. Log Daily Usage")
        print("2. Analyze Monthly Usage")
        print("3. Room Report")
        print("4. Export Monthly Report")
        print("5. Exit")
        choice = input("Choose option: ")

        if choice == "1":
            log_usage()
        elif choice == "2":
            analyze_usage()
        elif choice == "3":
            room_report()
        elif choice == "4":
            export_report()
        elif choice == "5":
            print("👋 Exiting...")
            break
        else:
            print("❌ Invalid choice! Try again.")


In [ ]:
if __name__ == "__main__":
    main()



--- Hostel Electricity Usage Logger ---
1. Log Daily Usage
2. Analyze Monthly Usage
3. Room Report
4. Export Monthly Report
5. Exit
